In [ ]:
import re
import string
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter, defaultdict

# Настройка стиля через matplotlib
plt.style.use('default')

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def letter_frequency(text):
    """Возвращает частоту букв в тексте (только A-Z)"""
    text = re.sub('[^A-Za-z]', '', text).upper()
    total = len(text)
    return {chr(i): text.count(chr(i)) / total if total > 0 else 0 
            for i in range(ord('A'), ord('Z')+1)}

def index_of_coincidence(text):
    """Вычисляет индекс совпадения (IC) для текста"""
    text = re.sub('[^A-Za-z]', '', text).upper()
    if len(text) < 2:
        return 0
    freq = Counter(text)
    n = len(text)
    ic = sum(f * (f - 1) for f in freq.values()) / (n * (n - 1))
    return ic

def caesar_decrypt(text, shift):
    """Расшифровка Цезаря со сдвигом"""
    result = ""
    for char in text:
        if 'A' <= char <= 'Z':
            result += chr((ord(char) - ord('A') - shift) % 26 + ord('A'))
        elif 'a' <= char <= 'z':
            result += chr((ord(char) - ord('a') - shift) % 26 + ord('a'))
        else:
            result += char
    return result

def frequency_score(text):
    """Оценивает текст по соответствию частотам английского языка"""
    english_freq = {
        'E': 0.127, 'T': 0.091, 'A': 0.082, 'O': 0.075, 'I': 0.070,
        'N': 0.067, 'S': 0.063, 'H': 0.061, 'R': 0.060, 'D': 0.043,
        'L': 0.040, 'C': 0.028, 'U': 0.028, 'M': 0.024, 'W': 0.023,
        'F': 0.022, 'G': 0.020, 'Y': 0.020, 'P': 0.019, 'B': 0.015,
        'V': 0.010, 'K': 0.008, 'J': 0.002, 'X': 0.001, 'Q': 0.001, 'Z': 0.001
    }
    text = text.upper()
    freq = letter_frequency(text)
    return sum(freq.get(k, 0) * english_freq.get(k, 0) for k in english_freq)

In [ ]:
def find_key_length_ic(ciphertext, max_key_length=20):
    """Определяет вероятную длину ключа через IC"""
    scores = []
    for key_len in range(1, max_key_length + 1):
        chunks = [''.join(ciphertext[i::key_len]) for i in range(key_len)]
        avg_ic = np.mean([index_of_coincidence(chunk) for chunk in chunks])
        scores.append((key_len, avg_ic))
    return scores

def plot_ic_analysis(ciphertext, max_key_length=20):
    """Визуализация IC для разных длин ключа"""
    scores = find_key_length_ic(ciphertext, max_key_length)
    key_lengths, ic_values = zip(*scores)
    
    plt.figure(figsize=(10, 5))
    plt.plot(key_lengths, ic_values, marker='o', linewidth=2)
    plt.axhline(y=0.067, color='r', linestyle='--', label='English IC (~0.067)')
    plt.axhline(y=0.038, color='g', linestyle='--', label='Random IC (~0.038)')
    plt.xlabel('Длина ключа')
    plt.ylabel('Индекс совпадения (IC)')
    plt.title('Анализ длины ключа шифра Виженера')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Возвращаем топ-3 кандидата
    top_candidates = sorted(scores, key=lambda x: x[1], reverse=True)[:3]
    print(f"Топ-3 кандидата на длину ключа: {top_candidates}")
    return top_candidates[0][0]

In [ ]:
def find_key_char(chunk):
    """Находит наиболее вероятный символ ключа для одного столбца"""
    best_shift = 0
    best_score = -1
    for shift in range(26):
        decrypted = caesar_decrypt(chunk, shift)
        score = frequency_score(decrypted)
        if score > best_score:
            best_score = score
            best_shift = shift
    return chr(best_shift + ord('A')), best_shift

def crack_vigenere(ciphertext, key_length):
    """Взламывает шифр при известной длине ключа"""
    chunks = [''.join(ciphertext[i::key_length]) for i in range(key_length)]
    key = ""
    shifts = []
    
    for i, chunk in enumerate(chunks):
        char, shift = find_key_char(chunk)
        key += char
        shifts.append(shift)
        print(f"Позиция {i+1}: сдвиг {shift} → '{char}'")
    
    # Полная расшифровка
    plaintext = ""
    for i, char in enumerate(ciphertext):
        if char.isalpha():
            shift = shifts[i % key_length]
            plaintext += caesar_decrypt(char, shift)
        else:
            plaintext += char
    return key, plaintext

In [ ]:
def plot_frequency_analysis(ciphertext, key_length):
    """График частот букв для каждого столбца (позиции ключа)"""
    chunks = [''.join(ciphertext[i::key_length]) for i in range(key_length)]
    letters = list(string.ascii_uppercase)
    
    fig, axes = plt.subplots(2, (key_length+1)//2, figsize=(15, 8))
    axes = axes.flatten()
    
    for idx, chunk in enumerate(chunks):
        freq = [chunk.upper().count(l) / len(re.sub('[^A-Za-z]', '', chunk)) if len(chunk) > 0 else 0 
                for l in letters]
        axes[idx].bar(letters, freq, color=sns.color_palette()[idx % 10])
        axes[idx].set_title(f'Позиция {idx+1}')
        axes[idx].tick_params(axis='x', rotation=90, labelsize=8)
        axes[idx].set_ylim(0, max(freq) * 1.2 if freq else 0.15)
    
    # Скрываем пустые подграфики
    for idx in range(len(chunks), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Частотный анализ по позициям ключа', fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_shift_heatmap(ciphertext, max_key_length=10):
    """Тепловая карта: строки = позиции в ключе, столбцы = сдвиги, цвет = скор"""
    heatmap_data = []
    
    for key_len in range(1, max_key_length + 1):
        chunks = [''.join(ciphertext[i::key_len]) for i in range(key_len)]
        row_scores = []
        for chunk in chunks:
            scores = []
            for shift in range(26):
                decrypted = caesar_decrypt(chunk, shift)
                scores.append(frequency_score(decrypted))
            row_scores.append(scores)
        # Усредняем по столбцам для визуализации
        if row_scores:
            avg_scores = np.mean(row_scores, axis=0)
            heatmap_data.append(avg_scores)
    
    # Дополняем до прямоугольной матрицы
    max_len = max(len(row) for row in heatmap_data)
    heatmap_data = [row + [0]*(max_len - len(row)) for row in heatmap_data]
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(heatmap_data[:max_key_length], 
                xticklabels=range(26),
                yticklabels=range(1, max_key_length+1),
                cmap='YlOrRd',
                cbar_kws={'label': 'Score (частотное соответствие)'})
    plt.xlabel('Сдвиг (0-25)')
    plt.ylabel('Длина ключа')
    plt.title('Тепловая карта подбора сдвигов')
    plt.tight_layout()
    plt.show()

In [ ]:
def crack_vigenere_full(ciphertext, max_key_length=20):
    """Полный автоматический взлом шифра Виженера"""
    print("🔍 Шаг 1: Анализ длины ключа...")
    estimated_length = plot_ic_analysis(ciphertext, max_key_length)
    
    print(f"\n🔑 Шаг 2: Подбор ключа (длина = {estimated_length})...")
    plot_frequency_analysis(ciphertext, estimated_length)
    
    key, plaintext = crack_vigenere(ciphertext, estimated_length)
    
    print(f"\n✅ Найденный ключ: '{key}'")
    print(f"\n📄 Расшифрованный текст (первые 500 символов):\n{plaintext[:500]}...")
    
    return key, plaintext

In [ ]:
# Пример зашифрованного текста (можно заменить на свой)
ciphertext = """
LXFOPVEFRNHR LXFOPVEFRNHR LXFOPVEFRNHR 
PYTHON IS AWESOME AND CRYPTOGRAPHY IS FUN 
ATTACK AT DAWN WITH THE NEW KEY
""".replace('\n', ' ')

# Запуск взлома
key, result = crack_vigenere_full(ciphertext, max_key_length=15)

# Проверка
print(f"\n🎯 Итог:")
print(f"Ключ: {key}")
print(f"Длина ключа: {len(key)}")